## Fine-tune Gemma-3 Task 2 (LoRA) — ver1_0_2
Aligned to A100 Colab template. Training hyperparameters unchanged.

---


### Quick Setup Instructions
* Run the installation cell below **once**.
* Click **"Restart session"** when it completes.
* Do not re-run install after restart.
* Use `huggingface-cli login` or Colab secrets for tokens (do not hardcode).


In [ ]:
import os
# Check if running in a Colab environment with GPU
if "COLAB_GPU" in os.environ:
    # Use unsloth[colab-new] for Colab to handle all dependencies.
    # This ensures compatible versions of bitsandbytes, accelerate, xformers, peft, trl, transformers etc.
    !pip install "unsloth[colab-new]==2026.1.3"
else:
    # For non-Colab environments or if no GPU, install basic unsloth.
    !pip install "unsloth==2026.1.3"


In [ ]:
from datasets import load_dataset
train_ds = load_dataset("Henryayay/POLCOVID_task2_70_10_20", split = "train")
validation_ds = load_dataset("Henryayay/POLCOVID_task2_70_10_20", split = "validation")
test_ds = load_dataset("Henryayay/POLCOVID_task2_70_10_20", split = "test")


In [ ]:
instruction = (
    "You are an expert radiologist. Analyze this chest X-ray and classify it as 'Non-COVID Pneumonia' or 'COVID-19'."

)



LABEL_MAP = {
    0: "Non-COVID Pneumonia",
    1: "COVID-19"
}


def convert_to_conversation(sample):
    label_text = LABEL_MAP[int(sample["label"])]

    conversation = [
    {
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "image", "image": sample["image"]},
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": label_text}
            ],
        },
    ]

    return {"messages": conversation}


In [ ]:
converted_dataset = [convert_to_conversation(sample) for sample in train_ds]
converted_val_dataset = [convert_to_conversation(sample) for sample in validation_ds]


In [ ]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-3-4b-it",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)


In [ ]:
from unsloth import get_chat_template

processor = get_chat_template(
    processor,
    "gemma-3"
)


In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 32,                           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 32,                  # Recommended alpha == r at least
    lora_dropout = 0.05,
    bias = "none",
    random_state = 3407,
    use_rslora = False,               # We support rank stabilized LoRA
    loftq_config = None,               # And LoftQ
    target_modules = "all-linear",    # Optional now! Can specify a list if needed
)


In [ ]:
from datasets import Dataset
from unsloth import FastVisionModel, UnslothVisionDataCollator
from unsloth.chat_templates import train_on_responses_only
from transformers import EarlyStoppingCallback
from trl import SFTTrainer, SFTConfig

# 1. Dataset Prep
train_dataset = Dataset.from_list(converted_dataset)
val_dataset = Dataset.from_list(converted_val_dataset)

# 2. Collator Prep
# Using 'processor' here ensures images are handled correctly
# 1. Define the Collator with masking built-in
tokenizer = processor.tokenizer
data_collator = UnslothVisionDataCollator(
    model = model,
    processor = processor,
    train_on_responses_only = True, # This replaces the need for the trainer function
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",

)


FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    data_collator = data_collator,
    processing_class = tokenizer, # Fix: Use processor, not tokenizer
    args = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        max_grad_norm = 0.3,
        warmup_ratio = 0.1,
        num_train_epochs = 10,
        learning_rate = 2e-4,
        logging_steps = 1,



        # --- Evaluation & Early Stopping ---
        eval_strategy = "epoch",
        save_strategy = "epoch",
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,

        # --- Hub Settings (Use a Secret, not hardcoded token!) ---
        output_dir = "outputs",
        push_to_hub = True,
        hub_model_id = "ccls1114/lightningAI_POLCOVID_FTGemma3_Task2-Checkpoints",
        hub_strategy = "every_save",
        hub_token = __import__("os").environ.get("HF_TOKEN"),  # set in Colab secrets
        save_total_limit = 2,

        optim = "adamw_torch_fused",
        weight_decay = 0.01, # Slightly higher for medical images
        lr_scheduler_type = "cosine",
        seed = 3407,

        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_seq_length = 2048,
   ),
    callbacks = [EarlyStoppingCallback(early_stopping_patience=5)] # 3 is usually enough
)



In [ ]:
# 1. Grab one sample from your dataset
sample = [trainer.train_dataset[0]]

# 2. Pass it through the collator manually
batch = data_collator(sample)

# 3. Now check the 'labels' inside that batch
labels = batch["labels"][0] # Get labels for the first item in batch

print("Tokens the model is actually learning from (Masking Check):")
print(tokenizer.decode([t for t in labels if t != -100]))


In [ ]:
trainer_stats = trainer.train()


In [ ]:
# ==========================================
# 1. SETUP & IMPORTS
# ==========================================
!pip install seaborn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch

# Ensure model is in inference mode (Unsloth method)
FastVisionModel.for_inference(model)

# Define your labels again just to be safe
LABEL_MAP = {
    0: "Non-COVID Pneumonia",
    1: "COVID-19",
}
# 2. INFERENCE LOOP (Test all 40 images)
# ==========================================
y_true = []
y_pred = []

print("Starting evaluation on test images...")

for sample in tqdm(test_ds):
    # A. Get the actual Label (Ground Truth)
    actual_label = LABEL_MAP[sample["label"]]
    y_true.append(actual_label)

    # B. Prepare the Image & Instruction
    image = sample["image"]
    instruction = (
    "You are an expert radiologist. Analyze this chest X-ray and classify it as 'Non-COVID Pneumonia' or 'COVID-19'."
)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "image"} # Placeholder
            ],
        }
    ]

    # C. Process Inputs
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(
        text=input_text,
        images=image,
        add_special_tokens=False,
        return_tensors="pt"
    ).to("cuda")
    # D. Generate Prediction
    output = model.generate(
        **inputs,
        max_new_tokens=15,
        temperature=0.01,   # Low temp = more strict/consistent
        use_cache=True
    )

    # E. Decode the answer
    # We slice [:, inputs.input_ids.shape[1]:] to remove the prompt and keep only the answer
    generated_text = processor.batch_decode(output[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

    # Clean up whitespace/newlines (e.g. "COVID19 " -> "COVID19")
    prediction = generated_text.strip()
    y_pred.append(prediction)

# ==========================================
# CALCULATE METRICS & PLOT
# ==========================================

# 1. Calculate Accuracy
acc = accuracy_score(y_true, y_pred)
accuracy_pct = acc * 100


# 2. Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=["Non-COVID Pneumonia", "COVID-19"])

# 3. Extract TN, FP, FN, TP
tn, fp, fn, tp = cm.ravel()

# 4. Calculate Sensitivity & Specificity
# Sensitivity = True Positive Rate (How well it finds Pneumonia)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0

# Specificity = True Negative Rate (How well it identifies Normal/Healthy)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# 5. Print Text Reports
print(f"\n✅ Fine Tuned Gemma3 Task2 Accuracy: {accuracy_pct:.2f}%")
print(f"🏥 Sensitivity: {sensitivity * 100:.2f}%")
print(f"🛡️ Specificity:  {specificity * 100:.2f}%")

print("\n📊 Detailed Report:")
print(classification_report(y_true, y_pred, labels=["Non-COVID Pneumonia", "COVID-19"]))

# 6. Plot Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Non-COVID Pneumonia", "COVID-19"],
            yticklabels=["Non-COVID Pneumonia", "COVID-19"])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f"Confusion Matrix\nFine Tuned Gemma3 Task2 (Accuracy: {accuracy_pct:.2f}%)")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 1. Get History
history = trainer.state.log_history

# 2. Extract Data
# Training Loss (Logged every 1 step)
train_steps = [x['step'] for x in history if 'loss' in x]
train_loss = [x['loss'] for x in history if 'loss' in x]

# Validation Loss (Logged every 20 steps)
val_steps = [x['step'] for x in history if 'eval_loss' in x]
val_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]

# 3. Plot
plt.figure(figsize=(10, 6))

# Plot Training Line
plt.plot(train_steps, train_loss, label="Training Loss", alpha=0.5, color="blue")

# Plot Validation Dots (The important ones)
plt.plot(val_steps, val_loss, label="Validation Loss", color="red", marker='o', linewidth=2)
plt.title("Task2: Gemma 3 Fine Tuning Dynamics (Early Stopping)")
plt.xlabel("Global Step")
plt.ylabel("Loss")

plt.ylim(0, 1.0) # Focus on the 0.0 to 1.0 range
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# 3. Plot
plt.figure(figsize=(10, 6))

# Plot Training Line
plt.plot(train_steps, train_loss, label="Training Loss", alpha=0.5, color="blue")

# Plot Validation Dots (The important ones)
plt.plot(val_steps, val_loss, label="Validation Loss", color="red", marker='o', linewidth=2)
plt.title("Task2: Gemma 3 Fine Tuning Dynamics (Early Stopping)")
plt.xlabel("Global Step")
plt.ylabel("Loss")

plt.ylim(0, 0.5) # Focus on the 0.0 to 0.5 range
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# 3. Plot
plt.figure(figsize=(10, 6))

# Plot Training Line
plt.plot(train_steps, train_loss, label="Training Loss", alpha=0.5, color="blue")

# Plot Validation Dots (The important ones)
plt.plot(val_steps, val_loss, label="Validation Loss", color="red", marker='o', linewidth=2)
plt.title("Task2: Gemma 3 Fine Tuning Dynamics (Early Stopping)")
plt.xlabel("Global Step")
plt.ylabel("Loss")

plt.ylim() # Focus on the 0.0 to 0.5 range
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator

# 1. Get History
history = trainer.state.log_history

# 2. Extract Data
# We now extract 'epoch' instead of 'step' for the X-axis
train_epochs = [x['epoch'] for x in history if 'loss' in x]
train_loss = [x['loss'] for x in history if 'loss' in x]

val_epochs = [x['epoch'] for x in history if 'eval_loss' in x]
val_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]

# 3. Plot
plt.figure(figsize=(10, 6))

# Plot Training Line (It will have many dots between integers, e.g. Epoch 0.1, 0.2...)
plt.plot(train_epochs, train_loss, label="Training Loss", alpha=0.5, color="blue")

# Plot Validation Dots (These will land exactly on integers: 1.0, 2.0, 3.0)
plt.plot(val_epochs, val_loss, label="Validation Loss", color="red", marker='o', linewidth=2)

plt.title("Task 2: Gemma 3 Fine-Tuning Dynamics (Early Stopping)")
plt.xlabel("Epoch")
plt.ylabel("Loss")

# --- THE FIX: Force X-axis to show only integers starting from 1 ---
# Get the max epoch number to set the limit
max_epoch = int(np.ceil(max(train_epochs)))
plt.xlim(0, max_epoch + 0.1) # Add a tiny buffer so the last dot isn't cut off
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
# -------------------------------------------------------------------

plt.ylim(0, 0.5) # Focus on the 0.0 to 0.5 range (adjust if needed)

plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Print the path of the best checkpoint
print(f"The Best Model was found at: {trainer.state.best_model_checkpoint}")

# Verify the metric
print(f"Best Validation Loss: {trainer.state.best_metric}")


In [ ]:
# Deprecated: predictions are saved via record_and_save_run() with
# Results/Fine-tune/TaskN/ and accuracy in the filename.


In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
candidates = sorted(Path('.').glob('Task2_FT_*_acc*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if not candidates and 'LOCAL_RESULTS_ROOT' in globals():
    candidates = sorted(LOCAL_RESULTS_ROOT.glob('Task2_FT_*_acc*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if candidates:
    display(FileLink(str(candidates[0])))
else:
    print('No Task2_FT_*_acc*.csv found yet. Run the evaluation/save cell first.')


### Batched Test Evaluation + Save Results (ver1_0_2)


In [ ]:
# === Aligned batched evaluation + results helper (ver1_0_2) ===
from pathlib import Path
import pandas as pd
import re
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from unsloth import FastVisionModel
import torch

LABEL_MAP = {0: "Non-COVID Pneumonia", 1: "COVID-19"}
NAME_TO_ID = {"Non-COVID Pneumonia": 0, "COVID-19": 1}
TASK_ID = "Task2_FT"
RESULTS_PROJECT_ROOT = Path(
    r"C:\Users\Henryayay\OneDrive - The Hong Kong Polytechnic University\Desktop\AI xray classification project\Results"
)
LOCAL_RESULTS_ROOT = RESULTS_PROJECT_ROOT / "Fine-tune" / "Task2"
PROMPT = "You are an expert radiologist. Analyze this chest X-ray and classify it as 'Non-COVID Pneumonia' or 'COVID-19'. Output only the diagnosis."

def _slug(text):
    text = str(text).strip().replace(" ", "-")
    return re.sub(r"[^A-Za-z0-9._-]+", "_", text)

def record_and_save_run(y_true, y_pred, setup, model_name, accuracy_pct, task_id=TASK_ID, trigger_download=True):
    y_true = list(y_true)
    y_pred = list(y_pred)
    df = pd.DataFrame({
        "Index": range(len(y_true)),
        "True_Label_ID": [NAME_TO_ID.get(str(v).strip(), -1) for v in y_true],
        "Predicted_Label_ID": [NAME_TO_ID.get(str(v).strip(), -1) for v in y_pred],
        "True_Label_Name": y_true,
        "Predicted_Label_Name": y_pred,
    })
    acc_label = f"{float(accuracy_pct):.2f}pct"
    filename = f"{_slug(task_id)}_{_slug(setup)}_{_slug(model_name)}_acc{acc_label}.csv"
    dest_dir = None
    for candidate in (LOCAL_RESULTS_ROOT, Path.cwd() / "Results" / "Fine-tune" / "Task2"):
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            probe = candidate / ".write_test"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            dest_dir = candidate
            break
        except Exception:
            continue
    if dest_dir is None:
        dest_dir = Path.cwd() / "Results" / "Fine-tune" / "Task2"
        dest_dir.mkdir(parents=True, exist_ok=True)
    dest_path = dest_dir / filename
    df.to_csv(dest_path, index=False)
    print(f"Saved predictions to: {dest_path}")
    if trigger_download and str(LOCAL_RESULTS_ROOT) not in str(dest_path):
        try:
            from google.colab import files
            print(f"If the browser save dialog appears, choose:\n{LOCAL_RESULTS_ROOT}")
            files.download(str(dest_path))
        except Exception as exc:
            print(f"Could not auto-download ({exc})")
    return dest_path

FastVisionModel.for_inference(model)
BATCH_SIZE = 8
y_true_batched, y_pred_batched = [], []
print(f"Starting batched FT evaluation (Batch Size: {BATCH_SIZE})...")

for i in tqdm(range(0, len(test_ds), BATCH_SIZE)):
    batch = test_ds[i : i + BATCH_SIZE]
    for lbl in batch["label"]:
        y_true_batched.append(LABEL_MAP[lbl])
    batch_messages, batch_images = [], []
    for img in batch["image"]:
        messages = [{
            "role": "user",
            "content": [
                {"type": "text", "text": PROMPT},
                {"type": "image"},
            ],
        }]
        batch_messages.append(processor.apply_chat_template(messages, add_generation_prompt=True))
        batch_images.append([img])
    inputs = processor(
        text=batch_messages,
        images=batch_images,
        add_special_tokens=False,
        return_tensors="pt",
        padding=True,
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=15, temperature=0.01, use_cache=True)
    prompt_len = inputs.input_ids.shape[1]
    responses = processor.batch_decode(outputs[:, prompt_len:], skip_special_tokens=True)
    for response in responses:
        if "COVID-19" in response:
            y_pred_batched.append("COVID-19")
        elif "Non-COVID Pneumonia" in response:
            y_pred_batched.append("Non-COVID Pneumonia")
        else:
            y_pred_batched.append("Error")

acc = accuracy_score(y_true_batched, y_pred_batched) * 100
print(f"\nBatched FT Task2 Accuracy: {acc:.2f}%")
print(classification_report(y_true_batched, y_pred_batched, labels=["Non-COVID Pneumonia", "COVID-19"]))
cm = confusion_matrix(y_true_batched, y_pred_batched, labels=["Non-COVID Pneumonia", "COVID-19"])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-COVID Pneumonia", "COVID-19"],
            yticklabels=["Non-COVID Pneumonia", "COVID-19"])
plt.title(f"FT Task2 Confusion Matrix (Batched)")
plt.show()
record_and_save_run(y_true_batched, y_pred_batched, setup="ft_batched", model_name="gemma-3-4b-it-lora", accuracy_pct=acc)
